In [1]:
import tarfile

# Extract the tar.gz file
with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall()

print("Extraction complete!")

/tmp/ipython-input-852/1308197893.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction complete!


In [3]:
import os
import shutil
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Clean up unused unsup folder (which breaks load_files)
unsup_path = os.path.join("aclImdb", "train", "unsup")
if os.path.exists(unsup_path):
    shutil.rmtree(unsup_path)

# 2. Load the dataset
print("Loading data...")
train_dir = os.path.join("aclImdb", "train")
test_dir = os.path.join("aclImdb", "test")

train_data = load_files(train_dir, categories=['pos', 'neg'], encoding='utf-8')
test_data = load_files(test_dir, categories=['pos', 'neg'], encoding='utf-8')
print(f"Loaded {len(train_data.data)} training samples and {len(test_data.data)} test samples.")

# 3. Generate the Document-Term Matrix
print("Vectorizing text...")
vectorizer = TfidfVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(train_data.data)
X_test = vectorizer.transform(test_data.data)

# 4. Train Naive Bayes Classifier
print("Training Multinomial Naive Bayes...")
clf = MultinomialNB(alpha=1.0)
clf.fit(X_train, train_data.target)

# 5. Evaluate the Model
print("Evaluating on Test Set...")
y_pred = clf.predict(X_test)
accuracy = accuracy_score(test_data.target, y_pred)
conf_matrix = confusion_matrix(test_data.target, y_pred)
report = classification_report(test_data.target, y_pred, target_names=test_data.target_names)

# 6. Output Results
print(f"\nFinal Classification Accuracy: {accuracy * 100:.4f}%")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nDetailed Accuracy Report:")
print(report)

Loading data...
Loaded 25000 training samples and 25000 test samples.
Vectorizing text...
Training Multinomial Naive Bayes...
Evaluating on Test Set...

Final Classification Accuracy: 82.9920%

Confusion Matrix:
[[10979  1521]
 [ 2731  9769]]

Detailed Accuracy Report:
              precision    recall  f1-score   support

         neg       0.80      0.88      0.84     12500
         pos       0.87      0.78      0.82     12500

    accuracy                           0.83     25000
   macro avg       0.83      0.83      0.83     25000
weighted avg       0.83      0.83      0.83     25000

